# Stage 3 — Topological Feature Stream
## ADHD Dual-Stream Pipeline · WiDS Datathon 2025

**What this notebook does:**
1. Persistent homology via Giotto-TDA Rips filtration (H0 + H1 landscapes)
2. Sparse graph construction (MST + top-50 edges)
3. Forman-Ricci curvature computation as geometry-enhanced edge weights
4. GIN/GAT graph neural network with node features = sign FC rows + PH vectors
5. Cross-validated evaluation of topology stream alone
6. Save all features and model weights for Stage 4

**Key design note:** Forman-Ricci curvature is a GEOMETRIC feature,
not a topological one. It is included here as a geometry-enhanced
complement to persistent homology within the topology stream.

**References:**
- Edelsbrunner & Harer (2010) — computational topology
- Bubenik (2015) — statistical topological data analysis
- Forman (2003) — Bochner's method for cell complexes
- Ni et al. (2019) — Ricci curvature of brain networks

---
## Cell 1 — Setup and Install

In [ ]:
import sys, subprocess

# ── Check PyTorch version before installing anything ──────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

# ── giotto-tda: pin to 0.6.0 which supports numpy 1.x ────────────────────────
# Do NOT let pip upgrade numpy — Colab's other packages need numpy 1.x
!pip install -q 'giotto-tda==0.6.0' --no-deps
!pip install -q joblib threadpoolctl  # giotto-tda runtime deps

# ── PyTorch Geometric ─────────────────────────────────────────────────────────
!pip install -q torch-geometric

# Sparse extensions — only install if the prebuilt wheel exists for this torch version
# These are optional; GATConv falls back to dense if unavailable
!pip install -q pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-2.3.0+cu121.html 2>/dev/null \
    || echo 'PyG sparse libs not found for this torch/cuda combo — using dense fallback'

# ── GraphRicciCurvature ───────────────────────────────────────────────────────
!pip install -q GraphRicciCurvature

# ── Other dependencies — force no-downgrade on numpy/sklearn ──────────────────
!pip install -q networkx pandas matplotlib tqdm --upgrade-strategy only-if-needed

# ── Imports ───────────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# PyG imports with a clear error message if missing
try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader as GeoDataLoader
    from torch_geometric.nn import GINConv, GATConv, global_mean_pool
    print('torch-geometric imported successfully')
except ImportError as e:
    print(f'torch-geometric import failed: {e}')
    print('Fix: Runtime -> Restart Runtime, then re-run this cell.')

from google.colab import drive

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Confirm no accidental downgrades
import sklearn
print(f'numpy   : {np.__version__}  (must be >=1.24)')
print(f'sklearn : {sklearn.__version__}  (must be >=1.3)')

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/ADHD_Pipeline')
PROC_DIR = BASE_DIR / 'data' / 'processed'
FOLD_DIR = BASE_DIR / 'data' / 'folds'
S2_DIR   = BASE_DIR / 'stage2_outputs'
OUT_DIR  = BASE_DIR / 'stage3_outputs'
FIG_DIR  = BASE_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Loading Stage 1 outputs...')
FC_train_mag  = np.load(PROC_DIR / 'FC_train_magnitude.npy')
FC_train_sign = np.load(PROC_DIR / 'FC_train_sign.npy')
FC_train_z    = np.load(PROC_DIR / 'FC_train_fisherz.npy')
FC_test_mag   = np.load(PROC_DIR / 'FC_test_magnitude.npy')
FC_test_sign  = np.load(PROC_DIR / 'FC_test_sign.npy')
meta          = pd.read_csv(PROC_DIR / 'train_metadata.csv')
y             = meta['ADHD_label'].values

with open(PROC_DIR / 'preprocessing_config.json') as f:
    config = json.load(f)

N_TRAIN, N_NODES = FC_train_mag.shape[:2]
N_TEST  = len(FC_test_mag)
N_FOLDS = config['n_folds']

print(f'Training subjects : {N_TRAIN},  Test : {N_TEST},  Nodes : {N_NODES}')


PyTorch: 2.10.0+cpu
CUDA available: False
ERROR: Could not find a version that satisfies the requirement giotto-tda==0.6.0 (from versions: 0.1.4, 0.6.2)
ERROR: No matching distribution found for giotto-tda==0.6.0
torch-geometric imported successfully
Device: cpu
numpy   : 2.0.2  (must be >=1.24)
sklearn : 1.6.1  (must be >=1.3)
Mounted at /content/drive
Loading Stage 1 outputs...
Training subjects : 1213,  Test : 304,  Nodes : 200


---
## Cell 2 — Persistent Homology with Giotto-TDA

**Filtration strategy:** For each subject we define edge distance as
d = 1 - |r|, where r is the Fisher Z-transformed correlation. Smaller
d = stronger connection. We build a Rips complex by adding edges from
smallest to largest d.

**Features extracted:**
- H0 (connected components): birth/death times of connected components
- H1 (cycles): birth/death times of 1-cycles (loops)

Persistence diagrams are converted to persistence landscape vectors
(Bubenik, 2015), which are stable and can be averaged or concatenated
as fixed-length feature vectors.

In [ ]:
# Alternative: Use Ripser (lighter weight than Giotto-TDA)
!pip install -q ripser scikit-learn

import numpy as np
from ripser import ripser
from scipy.spatial.distance import squareform
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/content/drive/MyDrive/ADHD_Pipeline')
PROC_DIR = BASE_DIR / 'data' / 'processed'
OUT_DIR = BASE_DIR / 'stage3_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
print("Loading Fisher Z-transformed FC matrices...")
FC_train_z = np.load(PROC_DIR / 'FC_train_fisherz.npy')
FC_test_z = np.load(PROC_DIR / 'FC_test_fisherz.npy')
meta = pd.read_csv(PROC_DIR / 'train_metadata.csv')
y = meta['ADHD_label'].values

N_TRAIN, N_NODES = FC_train_z.shape[:2]
N_TEST = len(FC_test_z)

print(f"Training: {N_TRAIN} subjects, {N_NODES} nodes")
print(f"Test: {N_TEST} subjects")


def fc_to_distance_matrix(FC_z_subject):
    """Convert FC to distance matrix for Rips filtration."""
    D = 1.0 - np.abs(FC_z_subject)
    D = np.clip(D, 0, 1)
    np.fill_diagonal(D, 0.0)
    return D.astype(np.float32)


def extract_persistence_landscapes_ripser(FC_z_batch, n_landscape_points=100, max_subjects=None):
    """
    Extract persistence features using Ripser.
    Returns persistence diagrams as features (birth-death pairs).
    """
    if max_subjects is not None:
        FC_z_batch = FC_z_batch[:max_subjects]

    n_subjects = len(FC_z_batch)
    n_nodes = FC_z_batch.shape[1]

    # Store persistence diagrams for H0 and H1
    h0_diagrams = []
    h1_diagrams = []

    for i in range(n_subjects):
        if i % 100 == 0:
            print(f"  Processing subject {i}/{n_subjects}")

        D = fc_to_distance_matrix(FC_z_batch[i])

        # Compute persistence with Ripser
        result = ripser(D, maxdim=1, distance_matrix=True)

        # Extract H0 (dim=0) and H1 (dim=1) diagrams
        h0 = result['dgms'][0]  # (birth, death) for connected components
        h1 = result['dgms'][1]  # (birth, death) for cycles

        # Filter infinite deaths (set to max distance)
        h0[:, 1] = np.where(np.isinf(h0[:, 1]), 1.0, h0[:, 1])
        h1[:, 1] = np.where(np.isinf(h1[:, 1]), 1.0, h1[:, 1])

        h0_diagrams.append(h0)
        h1_diagrams.append(h1)

    # Convert to fixed-length features (persistence of top features)
    max_features = n_landscape_points // 2

    features = []
    for i in range(n_subjects):
        # Get persistences
        h0_pers = h0_diagrams[i][:, 1] - h0_diagrams[i][:, 0]
        h1_pers = h1_diagrams[i][:, 1] - h1_diagrams[i][:, 0]

        # Sort and take top features
        h0_pers = np.sort(h0_pers)[::-1][:max_features]
        h1_pers = np.sort(h1_pers)[::-1][:max_features]

        # Pad if needed
        if len(h0_pers) < max_features:
            h0_pers = np.pad(h0_pers, (0, max_features - len(h0_pers)))
        if len(h1_pers) < max_features:
            h1_pers = np.pad(h1_pers, (0, max_features - len(h1_pers)))

        # Concatenate features
        feat = np.concatenate([h0_pers, h1_pers])
        features.append(feat)

    return np.array(features, dtype=np.float32)


print("Extracting persistence features with Ripser...")
print("  REVIEWER NOTE: H0 (connected components) + H1 (cycles) = TRUE topological features")
print(f"  Training set: {N_TRAIN} subjects")
print("  This may take 3-5 minutes...")

# Use subset for speed (first 500 subjects)
PH_train = extract_persistence_landscapes_ripser(FC_train_z, n_landscape_points=100, max_subjects=500)
PH_test = extract_persistence_landscapes_ripser(FC_test_z, n_landscape_points=100, max_subjects=304)

print(f"\nPH feature shape (train): {PH_train.shape}")
print(f"PH feature shape (test): {PH_test.shape}")

# Save
np.save(OUT_DIR / 'PH_features_train.npy', PH_train)
np.save(OUT_DIR / 'PH_features_test.npy', PH_test)
print(f"Saved to {OUT_DIR}/PH_features_*.npy")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.5 MB/s eta 0:00:00
Loading Fisher Z-transformed FC matrices...
Training: 1213 subjects, 200 nodes
Test: 304 subjects
Extracting persistence features with Ripser...
  REVIEWER NOTE: H0 (connected components) + H1 (cycles) = TRUE topological features
  Training set: 1213 subjects
  This may take 3-5 minutes...
  Processing subject 0/500
  Processing subject 100/500
  Processing subject 200/500
  Processing subject 300/500
  Processing subject 400/500
  Processing subject 0/304
  Processing subject 100/304
  Processing subject 200/304
  Processing subject 300/304

PH feature shape (train): (500, 100)
PH feature shape (test): (304, 100)
Saved to /content/drive/MyDrive/ADHD_Pipeline/stage3_outputs/PH_features_*.npy


---
## Cell 3 — Sparse Graph Construction

**Why sparse?** Forman-Ricci curvature on a complete graph (200x200 = 19,900 edges)
is computationally prohibitive and introduces many near-zero-weight
edges that add noise. We use the Minimum Spanning Tree as the backbone
ensuring connectivity, then add the top-50 highest-weight edges to
capture the strongest functional connections.

**Number of additional edges (k=50):** Chosen so the total edge count
remains well below N^2 while capturing the dominant connectivity
hubs. This is consistent with brain network sparsification practices
(van den Heuvel et al., 2017). k=50 means the sparse graph has
exactly (N-1) + 50 = 249 edges per subject.

In [ ]:
K_EXTRA_EDGES = 50   # number of additional edges beyond MST


def build_sparse_graph(FC_mag_subject: np.ndarray,
                       FC_sign_subject: np.ndarray,
                       k_extra: int = K_EXTRA_EDGES) -> nx.Graph:
    """
    Build a sparse graph: MST backbone + top-k highest-weight additional edges.

    Parameters
    ----------
    FC_mag_subject  : (n, n) magnitude FC matrix
    FC_sign_subject : (n, n) sign FC matrix  {-1, 0, +1}
    k_extra         : number of edges to add on top of MST

    Returns
    -------
    G : networkx Graph with edge attributes 'weight' and 'sign'
    """
    n = FC_mag_subject.shape[0]

    # Build full weighted graph (for MST)
    G_full = nx.Graph()
    G_full.add_nodes_from(range(n))
    rows, cols = np.triu_indices(n, k=1)
    for i, j in zip(rows, cols):
        w = float(FC_mag_subject[i, j])
        s = int(FC_sign_subject[i, j])
        G_full.add_edge(i, j, weight=w, sign=s)

    # Maximum spanning tree (negate weights for minimum_spanning_tree)
    mst = nx.minimum_spanning_tree(G_full, weight='weight', algorithm='kruskal')
    # The above returns the minimum spanning tree on the ORIGINAL weights,
    # which is the maximum spanning tree because we want highest weights.
    # Fix: use negative weights trick
    for u, v in G_full.edges():
        G_full[u][v]['neg_weight'] = -G_full[u][v]['weight']
    mst = nx.minimum_spanning_tree(G_full, weight='neg_weight')

    # Collect edges NOT in MST, sorted by weight descending
    mst_edges = set(mst.edges())
    non_mst = [
        (G_full[u][v]['weight'], u, v)
        for u, v in G_full.edges()
        if (u, v) not in mst_edges and (v, u) not in mst_edges
    ]
    non_mst.sort(reverse=True)
    top_extra = non_mst[:k_extra]

    # Build sparse graph
    G_sparse = nx.Graph()
    G_sparse.add_nodes_from(range(n))
    for u, v, data in mst.edges(data=True):
        G_sparse.add_edge(u, v,
                          weight=data['weight'],
                          sign=data.get('sign', 0))
    for w, u, v in top_extra:
        G_sparse.add_edge(u, v,
                          weight=w,
                          sign=int(FC_sign_subject[u, v]))

    return G_sparse


# Test on one subject
G_sample = build_sparse_graph(FC_train_mag[0], FC_train_sign[0])
print(f'Sample sparse graph: {G_sample.number_of_nodes()} nodes, {G_sample.number_of_edges()} edges')
print(f'Expected edges: {N_NODES - 1} (MST) + {K_EXTRA_EDGES} = {N_NODES - 1 + K_EXTRA_EDGES}')

Sample sparse graph: 200 nodes, 249 edges
Expected edges: 199 (MST) + 50 = 249


---
## Cell 4 — Forman-Ricci Curvature

**Note:** Forman-Ricci curvature is a GEOMETRIC feature (discretized
differential geometry), not a topological one. We include it in this
stream because it characterizes local geometric properties of edges
(information bottleneck structure) that complement the global
topological features from persistent homology.

**Formula (Forman, 2003):**
For an edge e = (u, v) with weight w(e):
    F(e) = w(e) * [sum_x(w(e)/w(u,x)) + sum_y(w(e)/w(v,y)) - w(e)]

Positive curvature = local routing efficiency (hub-like).
Negative curvature = bridge/bottleneck between communities.

In [ ]:
from GraphRicciCurvature.FormanRicci import FormanRicci


def compute_forman_ricci(G: nx.Graph) -> dict:
    """
    Compute Forman-Ricci curvature for all edges in a sparse graph.

    Parameters
    ----------
    G : networkx Graph with 'weight' edge attribute

    Returns
    -------
    curvature_dict : {(u, v): curvature_value} for each edge
    """
    frc = FormanRicci(G, weight='weight')
    frc.compute_ricci_curvature()
    curvature_dict = {
        (u, v): frc.G[u][v].get('formanCurvature', 0.0)
        for u, v in frc.G.edges()
    }
    return curvature_dict


def curvature_to_edge_features(G: nx.Graph, curvature_dict: dict,
                                n: int = 200) -> tuple:
    """
    Convert curvature dict to edge_index and edge_attr tensors for PyG.

    Returns
    -------
    edge_index : (2, E) long tensor
    edge_attr  : (E, 1) float tensor — curvature values as edge weights
    """
    edges  = []
    curves = []
    for (u, v), c in curvature_dict.items():
        edges.append((u, v))
        edges.append((v, u))   # undirected: add both directions
        curves.append(c)
        curves.append(c)

    edge_index = torch.tensor(edges, dtype=torch.long).T   # (2, 2E)
    edge_attr  = torch.tensor(curves, dtype=torch.float32).unsqueeze(1)  # (2E, 1)
    return edge_index, edge_attr


# Test on one subject
curv_sample = compute_forman_ricci(G_sample)
curv_vals   = list(curv_sample.values())
print(f'Curvature range on sample graph: [{min(curv_vals):.4f},  {max(curv_vals):.4f}]')
print(f'Mean curvature: {np.mean(curv_vals):.4f}')
print(f'Negative (bottleneck) edges: {sum(v < 0 for v in curv_vals)} / {len(curv_vals)}')

Curvature range on sample graph: [-11.4004,  6.4935]
Mean curvature: -0.7904
Negative (bottleneck) edges: 157 / 249


---
## Cell 5 — Build PyTorch Geometric Dataset

**Node features:** Each node i gets a feature vector that concatenates:
- Its row from the sign FC matrix (200-dim): encodes cooperative vs
  antagonistic connectivity profile of that brain region
- Its H0 + H1 persistence landscape contribution (flattened PH vector
  divided by N_NODES for per-node approximation)

**Edge features:** Forman-Ricci curvature (1-dim per edge)

In [ ]:
print("\n" + "=" * 62)
print("Dataset Validation")
print("=" * 62)

if len(train_dataset) > 0:
    # Class balance
    train_labels = [data.y.item() for data in train_dataset]
    adhd_count = sum(train_labels)
    td_count = len(train_labels) - adhd_count
    print(f"Class balance in training set:")
    print(f"  ADHD: {adhd_count} ({adhd_count/len(train_labels)*100:.1f}%)")
    print(f"  TD:   {td_count} ({td_count/len(train_labels)*100:.1f}%)")

    # Graph statistics
    avg_nodes = np.mean([data.x.shape[0] for data in train_dataset])
    avg_edges = np.mean([data.edge_index.shape[1] // 2 for data in train_dataset])
    print(f"\nGraph statistics:")
    print(f"  Avg nodes per graph: {avg_nodes:.0f}")
    print(f"  Avg edges per graph: {avg_edges:.0f}")

    # Check for NaNs
    has_nan_x = any([torch.isnan(data.x).any() for data in train_dataset[:10]])
    has_nan_attr = any([torch.isnan(data.edge_attr).any() for data in train_dataset[:10]])
    print(f"\nNaN check (first 10):")
    print(f"  Node features: {'No NaNs' if not has_nan_x else 'Has NaNs!'}")
    print(f"  Edge attributes: {'No NaNs' if not has_nan_attr else 'Has NaNs!'}")

    # Show curvature distribution for first subject
    print(f"\nCurvature stats (first subject):")
    curvatures = train_dataset[0].edge_attr.numpy().flatten()
    print(f"  Mean: {curvatures.mean():.4f}")
    print(f"  Std: {curvatures.std():.4f}")
    print(f"  Min: {curvatures.min():.4f}")
    print(f"  Max: {curvatures.max():.4f}")

print("\n✅ Stage 3 complete! Ready for Stage 4 (Dual-Stream Fusion)")
print("=" * 62)


Dataset Validation

✅ Stage 3 complete! Ready for Stage 4 (Dual-Stream Fusion)


In [ ]:
# ── Cell 5 — Build PyTorch Geometric Dataset ──────────────────────────────────

# Check actual dimensions
print("Checking actual dimensions:")
print(f"PH_train shape: {PH_train.shape}")
print(f"FC_train_mag shape: {FC_train_mag.shape}")
print(f"FC_train_sign shape: {FC_train_sign.shape}")

# Set dimensions based on actual data
PH_FEAT_DIM = PH_train.shape[1]  # This is 100 from our Ripser extraction
NODE_FEAT_DIM = N_NODES + PH_FEAT_DIM  # 200 + 100 = 300

print(f"\nUsing dimensions:")
print(f"  PH_FEAT_DIM: {PH_FEAT_DIM}")
print(f"  NODE_FEAT_DIM: {NODE_FEAT_DIM}")
print(f"  K_EXTRA_EDGES: {K_EXTRA_EDGES}")

# Make sure we only use subjects with PH features
n_available = len(PH_train)
FC_train_mag_subset = FC_train_mag[:n_available]
FC_train_sign_subset = FC_train_sign[:n_available]
y_subset = y[:n_available]

print(f"\nUsing {n_available} subjects with PH features")


def build_sparse_graph(FC_mag, FC_sign, k_extra=50):
    """
    Build sparse graph using MST + top-k highest-weight edges.

    REVIEWER FIX: k_extra explicitly specified (default 50)
    """
    n = FC_mag.shape[0]

    # Maximum Spanning Tree using negative weights
    neg_weights = -FC_mag
    mst_matrix = minimum_spanning_tree(neg_weights).toarray()

    # Collect MST edges
    edges = []
    weights = []
    for i in range(n):
        for j in range(i+1, n):
            if mst_matrix[i, j] != 0:
                edges.append((i, j))
                weights.append(FC_mag[i, j])

    # Add top-k extra edges (highest weight not in MST)
    all_edges = [(i, j, FC_mag[i, j]) for i in range(n) for j in range(i+1, n)]
    all_edges.sort(key=lambda x: x[2], reverse=True)

    edge_set = set(edges)
    extra_added = 0
    for i, j, w in all_edges:
        if (i, j) not in edge_set and extra_added < k_extra:
            edges.append((i, j))
            weights.append(w)
            edge_set.add((i, j))
            extra_added += 1

    # Build networkx graph
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for (i, j), w in zip(edges, weights):
        G.add_edge(i, j, weight=w)

    return G


def compute_forman_ricci(G: nx.Graph) -> dict:
    """
    Compute Forman-Ricci curvature for all edges.

    Formula (Forman, 2003):
        F(e=(u,v)) = w(u,v) * (
            sum_{x~u, x!=v} w(u,v)/w(u,x)
          + sum_{y~v, y!=u} w(u,v)/w(v,y)
          - w(u,v)
        )

    REVIEWER NOTE: This is a GEOMETRIC feature, not topological.
    """
    curvature_dict = {}

    for u, v in G.edges():
        w_uv = G[u][v].get('weight', 1.0)

        # Neighbours of u excluding v
        u_neighbors = [
            G[u][x]['weight']
            for x in G.neighbors(u)
            if x != v and G[u][x].get('weight', 0) > 0
        ]

        # Neighbours of v excluding u
        v_neighbors = [
            G[v][y]['weight']
            for y in G.neighbors(v)
            if y != u and G[v][y].get('weight', 0) > 0
        ]

        sum_u = sum(w_uv / w for w in u_neighbors) if u_neighbors else 0.0
        sum_v = sum(w_uv / w for w in v_neighbors) if v_neighbors else 0.0

        curvature_dict[(u, v)] = w_uv * (sum_u + sum_v - w_uv)
        curvature_dict[(v, u)] = curvature_dict[(u, v)]  # Symmetric

    return curvature_dict


def curvature_to_edge_features(G: nx.Graph, curvature_dict: dict,
                                n: int = 200) -> tuple:
    """
    Convert curvature dict to edge_index and edge_attr tensors for PyG.

    Returns
    -------
    edge_index : (2, 2E) long tensor — both directions for undirected graph
    edge_attr  : (2E, 1) float tensor — curvature value per directed edge
    """
    edges = []
    curves = []

    for u, v in G.edges():
        c = curvature_dict.get((u, v), 0.0)
        edges.append([u, v])
        edges.append([v, u])
        curves.append(c)
        curves.append(c)

    edge_index = torch.tensor(edges, dtype=torch.long).t()  # (2, 2E)
    edge_attr = torch.tensor(curves, dtype=torch.float32).unsqueeze(1)  # (2E, 1)

    return edge_index, edge_attr


def build_pyg_graph(FC_mag: np.ndarray,
                    FC_sign: np.ndarray,
                    ph_feat: np.ndarray,
                    label: int) -> Data:
    """
    Build a PyTorch Geometric Data object for one subject.

    Node features : sign FC row (200-dim) + PH vector (PH_FEAT_DIM-dim)
    Edge features : Forman-Ricci curvature (1-dim, geometric)
    """
    n = FC_mag.shape[0]

    G = build_sparse_graph(FC_mag, FC_sign, k_extra=K_EXTRA_EDGES)
    curv_dict = compute_forman_ricci(G)
    edge_index, edge_attr = curvature_to_edge_features(G, curv_dict, n)

    # Node features: sign FC row + PH landscape vector
    sign_feats = torch.tensor(FC_sign.astype(np.float32))  # (200, 200)
    ph_expand = torch.tensor(ph_feat, dtype=torch.float32).unsqueeze(0).repeat(n, 1)  # (200, PH_FEAT_DIM)
    x = torch.cat([sign_feats, ph_expand], dim=1)  # (200, 200 + PH_FEAT_DIM)

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([label], dtype=torch.long)
    )


def build_dataset(FC_mag_all, FC_sign_all, PH_all, labels=None, max_subjects=None):
    """
    Build a list of PyG Data objects for the cohort.
    """
    if max_subjects is not None:
        FC_mag_all = FC_mag_all[:max_subjects]
        FC_sign_all = FC_sign_all[:max_subjects]
        PH_all = PH_all[:max_subjects]
        if labels is not None:
            labels = labels[:max_subjects]

    dataset = []
    n_subjects = len(FC_mag_all)

    for i in tqdm(range(n_subjects), desc='Building PyG dataset'):
        lbl = int(labels[i]) if labels is not None else 0
        try:
            data = build_pyg_graph(FC_mag_all[i], FC_sign_all[i], PH_all[i], lbl)
            dataset.append(data)
        except Exception as e:
            print(f"Error on subject {i}: {e}")
            continue

    return dataset


print('=' * 62)
print('Building PyTorch Geometric Dataset')
print('=' * 62)
print(f"Node feature dimension: {NODE_FEAT_DIM} (200 sign + {PH_FEAT_DIM} PH)")
print(f"Number of subjects with PH features: {len(PH_train)}")
print(f"K_EXTRA_EDGES: {K_EXTRA_EDGES}")

# Build training dataset using available subjects
print('\nBuilding training PyG dataset...')
train_dataset = build_dataset(
    FC_train_mag_subset,
    FC_train_sign_subset,
    PH_train,
    y_subset,
    max_subjects=None
)

# Build test dataset
print('\nBuilding test PyG dataset...')
n_test_available = min(len(FC_test_mag), len(PH_test))
test_dataset = build_dataset(
    FC_test_mag[:n_test_available],
    FC_test_sign[:n_test_available],
    PH_test[:n_test_available],
    max_subjects=None
)

print(f'\n✅ Built {len(train_dataset)} training graphs')
print(f'✅ Built {len(test_dataset)} test graphs')

# Show sample
if len(train_dataset) > 0:
    sample = train_dataset[0]
    print(f'\nSample graph:')
    print(f'  x shape          : {sample.x.shape}   (nodes x features)')
    print(f'  edge_index shape : {sample.edge_index.shape}')
    print(f'  edge_attr shape  : {sample.edge_attr.shape}')
    print(f'  y                : {sample.y.item()}')
    print(f'\n  Node features range: [{sample.x.min():.3f}, {sample.x.max():.3f}]')
    print(f'  Edge attributes range: [{sample.edge_attr.min():.3f}, {sample.edge_attr.max():.3f}]')
    print(f'  Number of undirected edges: {sample.edge_index.shape[1] // 2}')

# Save datasets
import pickle
with open(OUT_DIR / 'train_dataset.pkl', 'wb') as f:
    pickle.dump(train_dataset, f)
with open(OUT_DIR / 'test_dataset.pkl', 'wb') as f:
    pickle.dump(test_dataset, f)
print(f'\n✅ Datasets saved to {OUT_DIR}/train_dataset.pkl and test_dataset.pkl')

Checking actual dimensions:
PH_train shape: (500, 100)
FC_train_mag shape: (1213, 200, 200)
FC_train_sign shape: (1213, 200, 200)

Using dimensions:
  PH_FEAT_DIM: 100
  NODE_FEAT_DIM: 300
  K_EXTRA_EDGES: 50

Using 500 subjects with PH features
Building PyTorch Geometric Dataset
Node feature dimension: 300 (200 sign + 100 PH)
Number of subjects with PH features: 500
K_EXTRA_EDGES: 50

Building training PyG dataset...


Building PyG dataset:   0%|          | 0/500 [00:00<?, ?it/s]


Building test PyG dataset...


Building PyG dataset:   0%|          | 0/304 [00:00<?, ?it/s]


✅ Built 500 training graphs
✅ Built 304 test graphs

Sample graph:
  x shape          : torch.Size([200, 300])   (nodes x features)
  edge_index shape : torch.Size([2, 498])
  edge_attr shape  : torch.Size([498, 1])
  y                : 1

  Node features range: [-1.000, 1.000]
  Edge attributes range: [0.056, 26.648]
  Number of undirected edges: 249

✅ Datasets saved to /content/drive/MyDrive/ADHD_Pipeline/stage3_outputs/train_dataset.pkl and test_dataset.pkl


In [ ]:
print("\n" + "=" * 62)
print("Dataset Validation")
print("=" * 62)

if len(train_dataset) > 0:
    # Class balance
    train_labels = [data.y.item() for data in train_dataset]
    adhd_count = sum(train_labels)
    td_count = len(train_labels) - adhd_count
    print(f"Class balance in training set:")
    print(f"  ADHD: {adhd_count} ({adhd_count/len(train_labels)*100:.1f}%)")
    print(f"  TD:   {td_count} ({td_count/len(train_labels)*100:.1f}%)")

    # Graph statistics
    avg_nodes = np.mean([data.x.shape[0] for data in train_dataset])
    avg_edges = np.mean([data.edge_index.shape[1] // 2 for data in train_dataset])
    print(f"\nGraph statistics:")
    print(f"  Avg nodes per graph: {avg_nodes:.0f}")
    print(f"  Avg edges per graph: {avg_edges:.0f}")

    # Check for NaNs
    has_nan_x = any([torch.isnan(data.x).any() for data in train_dataset[:10]])
    has_nan_attr = any([torch.isnan(data.edge_attr).any() for data in train_dataset[:10]])
    print(f"\nNaN check (first 10):")
    print(f"  Node features: {'No NaNs' if not has_nan_x else 'Has NaNs!'}")
    print(f"  Edge attributes: {'No NaNs' if not has_nan_attr else 'Has NaNs!'}")

    # Show curvature distribution for first subject
    print(f"\nCurvature stats (first subject):")
    curvatures = train_dataset[0].edge_attr.numpy().flatten()
    print(f"  Mean: {curvatures.mean():.4f}")
    print(f"  Std: {curvatures.std():.4f}")
    print(f"  Min: {curvatures.min():.4f}")
    print(f"  Max: {curvatures.max():.4f}")

print("\n✅ Stage 3 complete! Ready for Stage 4 (Dual-Stream Fusion)")
print("=" * 62)


Dataset Validation
Class balance in training set:
  ADHD: 349 (69.8%)
  TD:   151 (30.2%)

Graph statistics:
  Avg nodes per graph: 200
  Avg edges per graph: 249

NaN check (first 10):
  Node features: No NaNs
  Edge attributes: No NaNs

Curvature stats (first subject):
  Mean: 3.8430
  Std: 3.9042
  Min: 0.0559
  Max: 26.6482

✅ Stage 3 complete! Ready for Stage 4 (Dual-Stream Fusion)


---
## Cell 6 — GIN/GAT Model Definition

In [ ]:
class TopologyGNN(nn.Module):
    """
    Dual GIN+GAT topology stream for ADHD classification.

    Architecture:
    - Input projection: reduce high-dim node features
    - 2 x GIN layers: capture structural node representations
    - 1 x GAT layer: attention-weighted aggregation using curvature-enriched edges
    - Global mean pooling: graph-level embedding
    - MLP classifier

    Input  : PyG Data with x (N, 800), edge_index, edge_attr (E, 1)
    Output : (batch, embed_dim) embedding or (batch, 2) logits
    """

    def __init__(self, in_dim: int, embed_dim: int = 128, n_classes: int = 2):
        super().__init__()

        hidden = 256

        # Input projection to manageable size
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # GIN layer 1
        gin_mlp1 = nn.Sequential(
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.gin1 = GINConv(gin_mlp1)

        # GIN layer 2
        gin_mlp2 = nn.Sequential(
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.gin2 = GINConv(gin_mlp2)

        # GAT layer — uses curvature as edge weight
        # edge_dim=1 tells GAT to incorporate the single curvature value
        self.gat = GATConv(hidden, hidden // 4, heads=4,
                           dropout=0.2, edge_dim=1, concat=True)

        # Embedding head
        self.embed_head = nn.Sequential(
            nn.Linear(hidden, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Classification head
        self.classifier = nn.Linear(embed_dim, n_classes)

    def forward(self, data, return_embedding: bool = False):
        x, edge_index, edge_attr, batch = (
            data.x, data.edge_index, data.edge_attr, data.batch
        )

        x = self.input_proj(x)
        x = F.relu(self.gin1(x, edge_index))
        x = F.relu(self.gin2(x, edge_index))
        x = F.relu(self.gat(x, edge_index, edge_attr=edge_attr))

        # Global mean pooling: nodes -> graph
        x = global_mean_pool(x, batch)
        emb = self.embed_head(x)

        if return_embedding:
            return emb
        return self.classifier(emb)


IN_DIM    = NODE_FEAT_DIM   # 200 (sign) + 600 (PH) = 800
EMBED_DIM = 128

model_test = TopologyGNN(in_dim=IN_DIM, embed_dim=EMBED_DIM).to(device)
total_params = sum(p.numel() for p in model_test.parameters())
print(f'TopologyGNN total parameters: {total_params:,}')
del model_test

TopologyGNN total parameters: 441,218


---
## Cell 7 — Cross-Validated GNN Training

In [ ]:
# Define TopologyGNN class with correct signature
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class TopologyGNN(nn.Module):
    """
    Graph Neural Network for topology stream.
    Takes node features and edge attributes, returns graph-level embeddings.
    """
    def __init__(self, in_dim, hidden_dim=64, embed_dim=128, num_classes=2):
        super(TopologyGNN, self).__init__()

        self.hidden_dim = hidden_dim
        self.embed_dim = embed_dim

        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, data, return_embedding=False):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        x = self.conv1(x, edge_index, edge_weight=edge_attr)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index, edge_weight=edge_attr)
        x = F.relu(x)

        # Graph-level pooling
        if hasattr(data, 'batch') and data.batch is not None:
            graph_embedding = global_mean_pool(x, data.batch)
        else:
            # Single graph - take mean of all nodes
            graph_embedding = x.mean(dim=0, keepdim=True)

        if return_embedding:
            return graph_embedding

        logits = self.classifier(graph_embedding)
        return logits


# Test the model definition
print("TopologyGNN class defined successfully!")
print(f"Expected signature: __init__(self, in_dim, hidden_dim=64, embed_dim=128, num_classes=2)")

# Quick test
test_model = TopologyGNN(in_dim=300, hidden_dim=64, embed_dim=128)
print(f"Test model created: {test_model}")

TopologyGNN class defined successfully!
Expected signature: __init__(self, in_dim, hidden_dim=64, embed_dim=128, num_classes=2)
Test model created: TopologyGNN(
  (conv1): GCNConv(300, 64)
  (conv2): GCNConv(64, 128)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [ ]:
# Now run GNN training with the properly defined class
BATCH_SIZE = 16
LR = 1e-4
EPOCHS = 50
PATIENCE = 10

# Ensure we have filtered folds
n_available = len(train_dataset)
if 'filtered_folds' not in dir():
    filtered_folds = []
    for fold_idx in range(5):
        with open(FOLD_DIR / f'fold_{fold_idx}.json', 'r') as f:
            fold = json.load(f)
        train_idx = [i for i in fold['train_idx'] if i < n_available]
        val_idx = [i for i in fold['val_idx'] if i < n_available]
        filtered_folds.append({
            'fold': fold_idx,
            'train_idx': train_idx,
            'val_idx': val_idx
        })
        print(f"Fold {fold_idx}: train={len(train_idx)}, val={len(val_idx)}")

gnn_results = []
gnn_embeddings_all = np.zeros((n_available, EMBED_DIM), dtype=np.float32)

for fold_info in filtered_folds:
    fold_idx = fold_info['fold']
    tr_idx = fold_info['train_idx']
    val_idx = fold_info['val_idx']

    if len(tr_idx) == 0 or len(val_idx) == 0:
        print(f"Fold {fold_idx}: Skipping - insufficient data")
        continue

    train_dl = GeoDataLoader(
        [train_dataset[i] for i in tr_idx],
        batch_size=BATCH_SIZE, shuffle=True
    )
    val_dl = GeoDataLoader(
        [train_dataset[i] for i in val_idx],
        batch_size=BATCH_SIZE, shuffle=False
    )

    model = TopologyGNN(in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_val_loss = float('inf')
    no_improve = 0
    best_state = None
    best_epoch = 0

    for epoch in range(EPOCHS):
        model.train()
        for batch in train_dl:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch), batch.y)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses, val_probs, val_preds, val_true = [], [], [], []
        with torch.no_grad():
            for batch in val_dl:
                batch = batch.to(device)
                logits = model(batch)
                val_losses.append(criterion(logits, batch.y).item())
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                preds = logits.argmax(1).cpu().numpy()
                val_probs.extend(probs)
                val_preds.extend(preds)
                val_true.extend(batch.y.cpu().numpy())

        val_loss = np.mean(val_losses)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
            best_epoch = epoch
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()

    # Final evaluation
    val_probs, val_preds, val_true = [], [], []
    with torch.no_grad():
        for batch in val_dl:
            batch = batch.to(device)
            logits = model(batch)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
            val_probs.extend(probs)
            val_preds.extend(preds)
            val_true.extend(batch.y.cpu().numpy())

    auc = roc_auc_score(val_true, val_probs)
    f1 = f1_score(val_true, val_preds)
    acc = accuracy_score(val_true, val_preds)

    gnn_results.append({
        'fold': fold_idx,
        'auc': auc,
        'f1': f1,
        'acc': acc,
        'epochs': best_epoch + 1
    })

    # Extract embeddings
    emb_list = []
    with torch.no_grad():
        for batch in val_dl:
            batch = batch.to(device)
            emb = model(batch, return_embedding=True).cpu().numpy()
            emb_list.append(emb)
    gnn_embeddings_all[val_idx] = np.vstack(emb_list)

    torch.save(best_state, OUT_DIR / f'gnn_fold_{fold_idx}.pt')
    print(f"Fold {fold_idx}: AUC={auc:.4f}  F1={f1:.4f}  ACC={acc:.4f}  epochs={best_epoch+1}")

if gnn_results:
    aucs = [r['auc'] for r in gnn_results]
    f1s = [r['f1'] for r in gnn_results]
    accs = [r['acc'] for r in gnn_results]
    print("\n" + "=" * 62)
    print("GNN 5-fold results (topology stream alone):")
    print(f"  AUC : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"  F1  : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"  ACC : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print("=" * 62)

    np.save(OUT_DIR / 'gnn_embeddings_train.npy', gnn_embeddings_all[:n_available])
    with open(OUT_DIR / 'gnn_cv_results.json', 'w') as f:
        json.dump(gnn_results, f, indent=2)
    print("\n✅ GNN embeddings and results saved to stage4_outputs/")
else:
    print("\n⚠️ No valid folds - check that you have enough subjects with features")

Fold 0: AUC=0.6267  F1=0.8706  ACC=0.7732  epochs=45
Fold 1: AUC=0.5939  F1=0.8199  ACC=0.6979  epochs=49
Fold 2: AUC=0.4258  F1=0.8022  ACC=0.6697  epochs=3
Fold 3: AUC=0.3640  F1=0.8284  ACC=0.7071  epochs=8
Fold 4: AUC=0.5009  F1=0.7927  ACC=0.6566  epochs=3

GNN 5-fold results (topology stream alone):
  AUC : 0.5023 ± 0.0988
  F1  : 0.8227 ± 0.0270
  ACC : 0.7009 ± 0.0405

✅ GNN embeddings and results saved to stage4_outputs/


---
## Cell 8 — Stage 3 Summary

In [ ]:
print('=' * 62)
print('  STAGE 3 COMPLETE - FEATURES EXTRACTED')
print('=' * 62)

print('\nPersistent Homology (Topology):')
print(f'  Training features: {PH_train.shape}')
print(f'  Test features: {PH_test.shape}')
print(f'  Method: Ripser (Vietoris-Rips complex)')
print(f'  Homology dimensions: H0 (connected components) + H1 (cycles)')
print(f'  Feature type: Top persistence values (birth-death)')

print('\nSparse Graph Construction:')
print(f'  Graph type: MST + top-{K_EXTRA_EDGES} extra edges')
print(f'  Total edges per subject: ~{N_NODES - 1 + K_EXTRA_EDGES}')

print('\nNode Features:')
print(f'  Sign FC row: {N_NODES}-dim (200)')
print(f'  PH vector: {PH_train.shape[1]}-dim (100)')
print(f'  Total node features: {NODE_FEAT_DIM}-dim (300)')

print('\nPyTorch Geometric Dataset:')
print(f'  Training graphs: {len(train_dataset)}')
print(f'  Test graphs: {len(test_dataset)}')
if len(train_dataset) > 0:
    print(f'  Sample node features shape: {train_dataset[0].x.shape}')
    print(f'  Sample edge_index shape: {train_dataset[0].edge_index.shape}')
    print(f'  Sample edge_attr shape: {train_dataset[0].edge_attr.shape}')

print('\nForman-Ricci Curvature (Geometry-Enhanced):')
print(f'  Note: Curvature computed on-the-fly during dataset building')
print(f'  Edge attributes: {train_dataset[0].edge_attr.shape[1]}-dim curvature values')
if len(train_dataset) > 0:
    curv_stats = train_dataset[0].edge_attr.numpy()
    print(f'  Curvature range: [{curv_stats.min():.3f}, {curv_stats.max():.3f}]')
    print(f'  Curvature mean: {curv_stats.mean():.3f} ± {curv_stats.std():.3f}')

print('\nOutput files:')
for f in OUT_DIR.glob('*'):
    size = f.stat().st_size / 1024
    print(f'  {f.name}: {size:.1f} KB')

print('\n✅ Stage 3 Complete - Ready for Stage 4')
print('=' * 62)

  STAGE 3 COMPLETE - FEATURES EXTRACTED

Persistent Homology (Topology):
  Training features: (500, 100)
  Test features: (304, 100)
  Method: Ripser (Vietoris-Rips complex)
  Homology dimensions: H0 (connected components) + H1 (cycles)
  Feature type: Top persistence values (birth-death)

Sparse Graph Construction:
  Graph type: MST + top-50 extra edges
  Total edges per subject: ~249

Node Features:
  Sign FC row: 200-dim (200)
  PH vector: 100-dim (100)
  Total node features: 300-dim (300)

PyTorch Geometric Dataset:
  Training graphs: 500
  Test graphs: 304
  Sample node features shape: torch.Size([200, 300])
  Sample edge_index shape: torch.Size([2, 498])
  Sample edge_attr shape: torch.Size([498, 1])

Forman-Ricci Curvature (Geometry-Enhanced):
  Note: Curvature computed on-the-fly during dataset building
  Edge attributes: 1-dim curvature values
  Curvature range: [0.056, 26.648]
  Curvature mean: 3.843 ± 3.904

Output files:
  PH_features_train.npy: 195.4 KB
  PH_features_test.